# IT8414 – Tomato Leaf Disease Classification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eman-271/IT8414-Tomato-Leaf-Disease-Classification/blob/main/notebook/IT8414_Tomato_Leaf_Classification_with_Member2.ipynb) **Runtime → Run all** is all that is needed: Cell 1.0 downloads the data, the official split and the saved models from GitHub.
**Group [NN]** · Cells 1.0–5.0 author: **Fatema (Member 1 – Data Pipeline & Baseline Model)**

Classes: Healthy · Early Blight · Late Blight (PlantVillage). Run with **Runtime → Restart session and run all**.

| Sections | Owner | Content |
|---|---|---|
| 1.0 – 5.9 | Fatema (Member 1) | Setup, data, EDA (incl. class-choice evidence), preprocessing, split-ratio experiment, final split, baseline CNN, baseline tuning, baseline test evaluation |
| 6.0 – 7.0 | [Member 2] | Improved model, hyperparameter tuning, overfitting control |
| 8.0 – 10.0 | [Member 3] | Final test evaluation, error analysis, interpretability, conclusion |

**For teammates:** run sections 1.0–4.0 first. They load the *saved* split manifests, so every model uses exactly the same training, validation and test images.

### The whole pipeline at a glance

| Step | What happens | Cell | Output |
|---|---|---|---|
| 1 | Environment, fixed seed 42, Kaggle download unzipped | 1.0 | `data/original/` |
| 2 | The three tomato class folders found and counted | 2.0 | class counts |
| 3 | EDA: class balance, sample images, **colour evidence for the class choice** | 3.0 | Figures 3.1 – 3.3 |
| 4 | Balanced subset (100 per class, approved by the tutor), quality check | 4.0 | `data/selected/` |
| 5 | Preprocessing (decode → resize 224 → ÷255 → batch) and training-only augmentation | 4.0 | Figures 3.4 – 3.5 |
| 6 | Baseline CNN and the training helper with early stopping | 5.1 – 5.2 | – |
| 7 | **Split-ratio experiment**: 70/15/15 vs 80/10/10 vs 60/20/20, three random splits each, validation only | 5.3 | Figure 3.6, `split_ratio_comparison.csv` |
| 8 | Final stratified split with the chosen ratio, leakage checks, saved manifests | 5.4 | `data/split_manifests/` |
| 9 | Hyperparameter experiments on the final split (batch, learning rate, kernel), validation only | 5.5 – 5.6 | Figure D.1, results table |
| 10 | Final baseline: learning curves, saved model | 5.7 | Figure 5.1, `baseline_cnn_final.keras` |
| 11 | One-time test-set evaluation, bootstrap CIs, live-demo prediction | 5.8 | Figure 5.2, `baseline_test_results.json` |
| 12 | Summary, ready-to-paste numbers, download | 5.9 | `NUMBERS_FOR_REPORT.txt` |

## 1.0 Environment Setup

This section imports the libraries needed for data handling, visualization,
deep learning, and dataset splitting. It then sets up the project folders in
Colab (`data/`, `results/`) and gets the dataset in one of two ways:

- **Option A (default):** upload our dataset ZIP (`tomato_original.zip`, or the
  Kaggle download `archive.zip`) to `/content` (Files panel), or choose it when the
  upload button appears. **Teammates** can instead upload Member 1's share ZIP (any file named `*Member1_Share*.zip`):
  it is unzipped automatically and already holds the data, the saved split and the results. It is unzipped into
  `data/original/` and never changed.
- **Option B:** write the Kaggle dataset handle in `KAGGLE_DATASET`
  (for example `"owner/dataset-name"`), and the cell downloads it with `kagglehub`.

No Google Drive is needed.

**Offline (live demo laptop, no internet):** install `requirements.txt`, open this notebook from the
project's `notebook/` folder (or set the environment variable `IT8414_PROJECT_DIR` to the project
folder) and keep `tomato_original.zip` in the project folder. Cell 1.0 detects that it is not in
Colab, uses that folder instead of `/content`, and never tries to upload or download anything.

**Same numbers on every run:** GPU training is not bit-for-bit repeatable, so with
`M1_REUSE_SAVED = True` (Cell 5.2) the six tuning runs are loaded from `results/baseline_runs.json`
and the final model from `results/baseline_cnn_final.keras` whenever a saved run with exactly the
same settings exists. Change any setting (for example the learning rate in the live demo), or set
`M1_REUSE_SAVED = False`, and the model is trained again.

**Recommended:** Runtime → Change runtime type → **T4 GPU**, then
Runtime → **Run all**. Colab storage is temporary, so the last cell (5.9)
downloads the updated results.

A fixed random seed (42) is set with `tf.keras.utils.set_random_seed`, which
seeds Python, NumPy and TensorFlow together. The same seed is set again before
every model is built, so each experiment starts from the same weights and the
same data order.

In [ ]:
# Author: Fatema
# Cell 1.0 - Environment Setup

import os
import sys
import time
import random
import zipfile
import shutil
import json
from pathlib import Path
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
import tensorflow as tf
import keras

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Fixed random seed for reproducibility (seeds Python, NumPy and TensorFlow)
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# Show library versions (also recorded in README.md)
print("Python version:", sys.version.split()[0])
print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)
print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)
print("scikit-learn version:", sklearn.__version__)

# Check available hardware
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Author: Fatema
# Cell 1.0 - Project Folders and Data (Google Colab or an offline computer; nothing to upload)

import hashlib
import subprocess

try:
    from google.colab import files     # only exists in Google Colab
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

# Project folder. Colab: /content/IT8414_Tomato_Project. Any other computer (for example the
# offline demo laptop): the environment variable IT8414_PROJECT_DIR if it is set, otherwise the
# folder above notebook/ when the notebook is opened from the project's notebook/ folder.
if os.environ.get("IT8414_PROJECT_DIR"):
    PROJECT_DIR = Path(os.environ["IT8414_PROJECT_DIR"]).resolve()
elif IN_COLAB:
    PROJECT_DIR = Path("/content/IT8414_Tomato_Project")
elif Path.cwd().name == "notebook":
    PROJECT_DIR = Path.cwd().parent
else:
    PROJECT_DIR = Path.cwd() / "IT8414_Tomato_Project"
UPLOAD_DIR = Path("/content") if IN_COLAB else PROJECT_DIR.parent   # where uploaded ZIPs are looked for

DOWNLOAD_PATH = PROJECT_DIR / "data" / "original"         # unzipped Kaggle download (never changed)
SELECTED_PATH = PROJECT_DIR / "data" / "selected"         # balanced 100-per-class subset
SPLIT_PATH = PROJECT_DIR / "data" / "split_manifests"     # train / validation / test CSV files
RESULTS_PATH = PROJECT_DIR / "results"                    # saved model, figures and tables
FIGURES_PATH = RESULTS_PATH / "figures"

# ---- Get the project files automatically: nothing has to be uploaded ----
# The official 80/10/10 split is identified by the MD5 checksums of its three manifests. If the
# project folder does not hold exactly these files, the project is downloaded from the team's
# public GitHub repository (data, split manifests, saved models and results). A share ZIP that
# is already in the upload folder is tried first. A wrong or old folder is renamed, never used.
REPO_URL = "https://github.com/Eman-271/IT8414-Tomato-Leaf-Disease-Classification.git"
REPO_BRANCH = os.environ.get("IT8414_REPO_BRANCH", "main")
OFFICIAL_SPLIT_MD5 = {
    "train_split.csv": "cfee4ba34460dc51ff09b0f58c1b9b82",
    "validation_split.csv": "6b639291b1ade2753434ab390f150463",
    "test_split.csv": "62e2aabbf5121b62114908edd42aeb40"
}


def official_split_present(folder=PROJECT_DIR):
    manifests = folder / "data" / "split_manifests"
    return all((manifests / name).exists()
               and hashlib.md5((manifests / name).read_bytes().replace(b"\r\n", b"\n")).hexdigest() == md5
               for name, md5 in OFFICIAL_SPLIT_MD5.items())


if not official_split_present():
    # 1. A share ZIP that is already there (optional; the newest one is used)
    share_zips = sorted(UPLOAD_DIR.glob("*_Share*.zip"), key=lambda z: z.stat().st_mtime, reverse=True)
    if share_zips and not PROJECT_DIR.exists():
        with zipfile.ZipFile(share_zips[0]) as z:
            z.extractall(UPLOAD_DIR)
        print("Share package unzipped from:", share_zips[0].name)

if not official_split_present():
    # 2. Download the project from GitHub (about 150 MB, under a minute in Colab)
    if PROJECT_DIR.exists():
        old = PROJECT_DIR.with_name(PROJECT_DIR.name + "_old_" + time.strftime("%Y%m%d_%H%M%S"))
        PROJECT_DIR.rename(old)
        print("The project folder did not hold the official split; moved it aside to:", old)
    print("Downloading the project from GitHub:", REPO_URL)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_DIR)],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError("Could not download the project from GitHub (no internet?).\n" + clone.stderr +
                           "\nOffline: copy the project folder (a clone of the repository) to this computer.")

if not official_split_present():
    raise RuntimeError("The official 80/10/10 split manifests are missing or changed in " + str(PROJECT_DIR))
print("Official 80/10/10 split manifests found (checksums match).")

for folder in [SPLIT_PATH, RESULTS_PATH, FIGURES_PATH]:
    folder.mkdir(parents=True, exist_ok=True)

# Option B: Kaggle dataset handle, e.g. "owner/dataset-name". Leave empty to use a ZIP (Option A).
# Option B needs internet, so leave it empty on the offline demo laptop.
KAGGLE_DATASET = ""

DOWNLOAD_PATH.mkdir(parents=True, exist_ok=True)
has_images = any(f.suffix.lower() in {".jpg", ".jpeg", ".png"} for f in DOWNLOAD_PATH.rglob("*"))

if not has_images:
    if KAGGLE_DATASET:
        # Option B: download with kagglehub and copy it into data/original
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        shutil.copytree(downloaded, DOWNLOAD_PATH / "kaggle_download", dirs_exist_ok=True)
        print("Dataset downloaded from Kaggle:", KAGGLE_DATASET)
    else:
        # Option A: unzip the dataset ZIP once (data/original/tomato_original.zip comes with the
        # repository; an uploaded tomato_original.zip or Kaggle archive.zip also works)
        search_dirs = [DOWNLOAD_PATH, PROJECT_DIR] + ([UPLOAD_DIR] if IN_COLAB else [])
        zips = [z for d in search_dirs for z in sorted(d.glob("*.zip"))
                if not any(word in z.name.lower() for word in ("results", "member", "share"))]
        if not zips and IN_COLAB:
            print("Please upload the dataset ZIP (tomato_original.zip or the Kaggle archive.zip)")
            uploaded = files.upload()
            zips = [UPLOAD_DIR / name for name in uploaded]
        if not zips:
            raise FileNotFoundError(f"Put tomato_original.zip into {PROJECT_DIR} and run this cell again.")
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(DOWNLOAD_PATH)
        print("Dataset unzipped from:", zips[0].name)

print("Running in:", "Google Colab" if IN_COLAB else "local / offline mode")
print("Project folder:", PROJECT_DIR)


## 2.0 Dataset Loading and Verification

This section loads the original tomato leaf images and verifies the selected
classes, number of images, and image file formats before preprocessing.

**Dataset source and licence**
- Original collection: PlantVillage (Hughes & Salathé, 2015).
- Downloaded from Kaggle: *plantvillage Tomato leaf dataset* by charu (2019) –
  https://www.kaggle.com/datasets/charuchaudhry/plantvillage-tomato-leaf-dataset
- Licence: **Apache License 2.0** (http://www.apache.org/licenses/LICENSE-2.0),
  the licence of the PlantVillage dataset, of which these tomato classes are part.
  The tutor confirmed that this dataset and licence may be used for the project. The data is used for
  non-commercial academic work only, the images are not redistributed with the
  project (only a download link and preparation steps), and the original
  PlantVillage authors are credited. Full details are in `documentation/DATA.md`.

The next cell lists every class folder in the download and then **finds the three
tomato folders automatically**, whatever their exact names are (for example
`Tomato___Early_blight`), so the cell still works if the folders are renamed.

**Why these three classes?**
- **Early Blight** (*Alternaria solani*) and **Late Blight** (*Phytophthora infestans*)
  both produce brown, dead (necrotic) patches on the leaf, so they look similar.
  Separating them is a fine-grained problem and makes the task genuinely challenging.
- **Healthy** leaves are the reference class: a screening tool must first say
  whether a leaf is diseased at all.
- Using three classes makes this a **multi-class** problem with one easy
  separation (healthy vs diseased) and one hard one (Early vs Late Blight).
- This choice is **checked with evidence from the images** in 3.0 (Figure 3.3):
  the colour of the two blights is measured and compared with Healthy leaves.

The selected classes are:
- Tomato Healthy
- Tomato Early Blight
- Tomato Late Blight

In [ ]:
# Author: Fatema
# Cell 2.0 - Dataset Loading and Verification

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
CLASS_NAMES = ["Healthy", "Early Blight", "Late Blight"]   # label IDs 0, 1, 2


def count_images(folder):
    return len([f for f in folder.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS])


def matches_class(folder_name, class_name):
    # Recognise the tomato class from the folder name (works with _ or ___ in the name)
    name = folder_name.lower()
    if "tomato" not in name:
        return False
    if class_name == "Healthy":
        return "healthy" in name
    if class_name == "Early Blight":
        return "early" in name and "blight" in name
    return "late" in name and "blight" in name


def find_class_folders():
    # Every folder in the download that holds images of one of the three classes
    all_folders = [d for d in DOWNLOAD_PATH.rglob("*") if d.is_dir()]
    candidates = {name: [d for d in all_folders if matches_class(d.name, name) and count_images(d) > 0]
                  for name in CLASS_NAMES}
    for name, found in candidates.items():
        if not found:
            raise FileNotFoundError(f"No tomato {name} folder found in the download")

    # The three classes must come from the same parent folder. Some Kaggle archives
    # contain the data twice; then the copy with the most images is used.
    shared_parents = set.intersection(*[{d.parent for d in found} for found in candidates.values()])
    if not shared_parents:
        raise FileNotFoundError("The three tomato class folders are not in one folder")

    def folders_in(parent):
        return [next(d for d in candidates[name] if d.parent == parent) for name in CLASS_NAMES]

    best_parent = max(sorted(shared_parents), key=lambda p: sum(count_images(d) for d in folders_in(p)))
    if len(shared_parents) > 1:
        print(f"Note: the download holds {len(shared_parents)} copies; using {best_parent.relative_to(DOWNLOAD_PATH)}")
    return folders_in(best_parent)


class_folders = find_class_folders()

ORIGINAL_PATH = class_folders[0].parent
EXPECTED_CLASSES = [folder.name for folder in class_folders]   # real folder names, in label order
HAS_ORIGINAL = True

# Every class in the download (the three tomato classes are a subset)
all_classes = sorted(d.name for d in ORIGINAL_PATH.iterdir() if d.is_dir() and count_images(d) > 0)
print(f"Class folders in the download: {len(all_classes)}")
for name in all_classes:
    marker = "  <- used" if name in EXPECTED_CLASSES else ""
    print(f"   {name}: {count_images(ORIGINAL_PATH / name)} images{marker}")

print("\nDataset folder:", ORIGINAL_PATH.relative_to(PROJECT_DIR))
for display_name, folder in zip(CLASS_NAMES, class_folders):
    print(f"{display_name:13s} -> {folder.name}: {count_images(folder)} images")

# File formats of the three selected classes
formats = pd.Series([f.suffix.lower() for folder in class_folders for f in folder.iterdir()
                     if f.suffix.lower() in IMAGE_EXTENSIONS]).value_counts()
print("\nFile formats:", formats.to_dict())

## 3.0 Exploratory Data Analysis (EDA)

This section explores the original dataset before preprocessing.
The class distribution is visualized to identify whether the dataset is balanced
and to support later decisions about dataset selection and splitting.
Random sample images are shown to inspect image quality, backgrounds and
the visual difference between the classes. A third analysis measures the
colour of the leaves to **check the class choice with evidence** (Figure 3.3).

In [ ]:
# Author: Fatema
# Cell 3.0 - EDA: Class Distribution

if not HAS_ORIGINAL:
    print("Skipped: original images not available. Member 1 ran this EDA - see results/figures/fig3_1_class_distribution.png")
else:

    class_counts = {}

    for class_name in EXPECTED_CLASSES:
        class_path = ORIGINAL_PATH / class_name

        image_files = [
            file for file in class_path.iterdir()
            if file.suffix.lower() in IMAGE_EXTENSIONS
        ]

        class_counts[class_name] = len(image_files)

    # Print counts, share of the dataset and imbalance ratio
    total_images = sum(class_counts.values())

    for class_name, count in class_counts.items():
        print(f"{class_name}: {count} images ({count / total_images:.1%})")

    print("Imbalance ratio (largest / smallest class):",
          round(max(class_counts.values()) / min(class_counts.values()), 2))

    # Create a simple bar chart
    plt.figure(figsize=(8, 5))

    plt.bar(CLASS_NAMES, [class_counts[c] for c in EXPECTED_CLASSES])

    plt.title("Original Dataset Class Distribution")
    plt.xlabel("Tomato Leaf Class")
    plt.ylabel("Number of Images")
    plt.savefig(FIGURES_PATH / "fig3_1_class_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Author: Fatema
# Cell 3.0 - EDA: Multiple Sample Images

if not HAS_ORIGINAL:
    print("Skipped: original images not available. Member 1 ran this EDA - see results/figures/fig3_2_sample_images.png")
else:

    samples_per_class = 4
    sample_rng = random.Random(SEED)

    plt.figure(figsize=(12, 9))

    plot_position = 1

    for class_name in EXPECTED_CLASSES:
        class_path = ORIGINAL_PATH / class_name

        image_files = sorted([
            file for file in class_path.iterdir()
            if file.suffix.lower() in IMAGE_EXTENSIONS
        ])

        # Randomly choose sample images using the fixed seed
        sample_files = sample_rng.sample(image_files, samples_per_class)

        for image_path in sample_files:
            image = tf.keras.utils.load_img(image_path)

            plt.subplot(3, samples_per_class, plot_position)
            plt.imshow(image)

            display_name = CLASS_NAMES[EXPECTED_CLASSES.index(class_name)]
            plt.title(display_name)
            plt.axis("off")

            plot_position += 1

    plt.tight_layout()
    plt.savefig(FIGURES_PATH / "fig3_2_sample_images.png", dpi=150, bbox_inches="tight")
    plt.show()

### 3.0 Why Early Blight and Late Blight? Evidence from the images

The two diseases were chosen because they **look alike**, which makes the task a
fine-grained problem rather than a simple "green vs brown" decision. This claim
is tested here with a simple, transparent measurement on 150 random images per
class (seed 42):

1. **Leaf mask:** pixels that are clearly coloured (saturation > 0.20) and not dark
   (brightness > 0.15) are treated as leaf; the grey or black background is ignored.
2. **Lesion fraction:** the share of leaf pixels whose hue is brown/yellow
   (10°–55°). Healthy tissue is green (about 60°–150°).
3. **Colour profile:** an 18-bin hue histogram of the leaf pixels. The distance
   between two classes is half the sum of absolute differences between their
   average histograms (0 = identical colours, 1 = no colour in common).
4. **Colour-only classifier:** each image is assigned to the class whose average
   histogram is closest. If colour alone could separate the classes, a CNN would
   not be needed; where it fails shows which classes are genuinely hard.

In [ ]:
# Author: Fatema
# Cell 3.0 - Evidence for the Class Choice: Lesion Colour and Class Similarity

N_EVIDENCE = 150   # random images per class
evidence_rng = random.Random(SEED)


def leaf_colour_features(image_path):
    # Convert to HSV (hue, saturation, value) at a small size for speed
    with Image.open(image_path) as img:
        hsv = np.asarray(img.convert("RGB").resize((128, 128)).convert("HSV"), dtype=np.float32)
    hue = hsv[..., 0] * 360 / 255          # PIL stores hue as 0-255 -> convert to degrees
    saturation = hsv[..., 1] / 255
    brightness = hsv[..., 2] / 255

    leaf_hue = hue[(saturation > 0.20) & (brightness > 0.15)]      # leaf pixels only
    lesion_fraction = float(np.mean((leaf_hue >= 10) & (leaf_hue <= 55))) if leaf_hue.size else 0.0
    histogram, _ = np.histogram(leaf_hue, bins=18, range=(0, 180))
    return lesion_fraction, histogram / max(histogram.sum(), 1)


lesion_fractions, hue_histograms = {}, {}
for folder_name, display_name in zip(EXPECTED_CLASSES, CLASS_NAMES):
    image_files = sorted(f for f in (ORIGINAL_PATH / folder_name).iterdir()
                         if f.suffix.lower() in IMAGE_EXTENSIONS)
    sample_files = evidence_rng.sample(image_files, min(N_EVIDENCE, len(image_files)))
    features = [leaf_colour_features(f) for f in sample_files]
    lesion_fractions[display_name] = np.array([f[0] for f in features])
    hue_histograms[display_name] = np.array([f[1] for f in features])

mean_histograms = {name: h.mean(axis=0) for name, h in hue_histograms.items()}
colour_distance = lambda a, b: 0.5 * np.abs(a - b).sum()

# 1. Lesion fraction per class
print("Brown/yellow (lesion-coloured) share of the leaf, mean over", N_EVIDENCE, "images:")
for name in CLASS_NAMES:
    print(f"   {name:13s} {lesion_fractions[name].mean():.1%}")

# 2. Colour distance between every pair of classes
pair_distances = {}
print("\nColour-profile distance between classes (0 = identical, 1 = completely different):")
for i in range(3):
    for j in range(i + 1, 3):
        a, b = CLASS_NAMES[i], CLASS_NAMES[j]
        pair_distances[f"{a} - {b}"] = round(float(colour_distance(mean_histograms[a], mean_histograms[b])), 3)
        print(f"   {a} - {b}: {pair_distances[f'{a} - {b}']}")
closest_pair = min(pair_distances, key=pair_distances.get)
print("Most similar pair:", closest_pair)

# 3. Colour-only nearest-average classifier
colour_cm = np.zeros((3, 3), dtype=int)
for i, name in enumerate(CLASS_NAMES):
    for h in hue_histograms[name]:
        distances = [colour_distance(h, mean_histograms[other]) for other in CLASS_NAMES]
        colour_cm[i, int(np.argmin(distances))] += 1
colour_accuracy = np.trace(colour_cm) / colour_cm.sum()
print(f"\nColour-only classifier accuracy: {colour_accuracy:.1%} (rows = true class, columns = predicted)")
print(pd.DataFrame(colour_cm, index=CLASS_NAMES, columns=CLASS_NAMES))

# Figure 3.3: lesion fraction (left) and average colour profile (right)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot([lesion_fractions[n] * 100 for n in CLASS_NAMES], showfliers=False)
axes[0].set_xticks([1, 2, 3], CLASS_NAMES)
axes[0].set_title("Share of the Leaf with Brown/Yellow (Lesion) Colour")
axes[0].set_ylabel("% of leaf pixels (hue 10°-55°)")
axes[0].grid(True, axis="y")

bin_centres = np.arange(5, 180, 10)
for name in CLASS_NAMES:
    axes[1].plot(bin_centres, mean_histograms[name], marker="o", label=name)
axes[1].axvspan(10, 55, color="peru", alpha=0.15, label="Lesion colours (10°-55°)")
axes[1].set_title("Average Colour Profile of the Leaf (hue histogram)")
axes[1].set_xlabel("Hue (degrees)")
axes[1].set_ylabel("Share of leaf pixels")
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "fig3_3_class_similarity.png", dpi=150, bbox_inches="tight")
plt.show()

class_evidence = {
    "images_per_class": N_EVIDENCE,
    "mean_lesion_fraction": {n: round(float(lesion_fractions[n].mean()), 4) for n in CLASS_NAMES},
    "colour_distance": pair_distances,
    "colour_only_confusion_matrix": colour_cm.tolist(),
    "colour_only_accuracy": round(float(colour_accuracy), 4),
}
with open(RESULTS_PATH / "class_choice_evidence.json", "w") as f:
    json.dump(class_evidence, f, indent=2)

**What the evidence shows** *(the measured values are printed above; on the
PlantVillage tomato images they were Healthy ≈ 1%, Early Blight ≈ 15% and Late
Blight ≈ 19% lesion-coloured leaf, with Early – Late Blight the most similar pair)*

- **Both blights damage the leaf in the same colours.** A large share of their
  leaf area is brown/yellow, while Healthy leaves have almost none. Colour tells a
  diseased leaf from a healthy one, but it does not tell *which* blight it is.
- **Early and Late Blight are the most similar pair** of the three: the distance
  between their colour profiles is the smallest.
- **A colour-only classifier fails on exactly the classes we chose.** It
  recognises most Healthy leaves, but confuses Early Blight with Late Blight, and
  many Late Blight leaves with Healthy (Late Blight lesions are often small and
  pale near the leaf edge, so the leaf stays mostly green).

**Decision:** the class choice makes the task genuinely fine-grained. The model
must learn the **shape and texture** of the lesions (Early Blight: dark spots
with concentric rings; Late Blight: large irregular, water-soaked patches), not
just their colour. This also tells us what to look for later: most test errors
are expected **between the two blights**, and **Late Blight → Healthy** is the
costliest mistake to watch (5.8).

## 4.0 Data Preparation and Splitting

The EDA showed that the original classes are imbalanced (see the imbalance ratio
printed in 3.0). A balanced subset is therefore used:
100 images are selected from each class with a fixed random seed. This removes
class imbalance from the model comparison and keeps each training run short
enough to repeat for the hyperparameter experiments.
The 300-image balanced subset was approved by the tutor at the proposal stage.

The selected dataset contains 300 images in total:
- 100 Healthy
- 100 Early Blight
- 100 Late Blight

Steps in this section:
1. Copy the balanced subset (the original images are never modified).
2. Check that every selected image opens correctly and has the same size.
3. Define a **stratified split function** (`make_split`). The split ratio is not
   assumed: three ratios (70/15/15, 80/10/10, 60/20/20) are compared by experiment
   in **5.3**, once the baseline model exists, and the final split, the leakage
   checks and the saved manifests follow in **5.4**.
4. Build `tf.data` pipelines (decode → resize → normalise → batch).
5. Define training-only data augmentation.
6. Draw the complete data pipeline (Figure 3.4).

In [ ]:
# Author: Fatema
# Cell 4.0 - Balanced Dataset Selection

IMAGES_PER_CLASS = 100

# Set to True only if the balanced subset must be recreated from scratch.
# The selection is seeded, so recreating it gives exactly the same 300 images.
REBUILD_SELECTED = False


def selected_subset_complete():
    # True when every class folder already holds 100 images
    if not SELECTED_PATH.exists():
        return False
    for class_name in EXPECTED_CLASSES:
        class_path = SELECTED_PATH / class_name
        if not class_path.exists():
            return False
        n_images = len([f for f in class_path.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS])
        if n_images != IMAGES_PER_CLASS:
            return False
    return True


if selected_subset_complete() and not REBUILD_SELECTED:
    print("Balanced subset already exists - using the saved selection.")
    CLASSES_TO_BUILD = []
elif not HAS_ORIGINAL:
    raise FileNotFoundError(
        "data/selected is incomplete and data/original is missing. "
        "Ask Member 1 for the full shared ZIP, or download the Kaggle data (see documentation/DATA.md).")
else:
    # Remove any partial or old selection and build it again
    if SELECTED_PATH.exists():
        shutil.rmtree(SELECTED_PATH)
    SELECTED_PATH.mkdir(parents=True)
    CLASSES_TO_BUILD = EXPECTED_CLASSES

selection_rng = random.Random(SEED)

for class_name in CLASSES_TO_BUILD:

    source_path = ORIGINAL_PATH / class_name
    destination_path = SELECTED_PATH / class_name

    # Create a folder for each class
    destination_path.mkdir(exist_ok=True)

    # Find all valid image files (sorted so the selection is reproducible)
    image_files = sorted([
        file for file in source_path.iterdir()
        if file.suffix.lower() in IMAGE_EXTENSIONS
    ])

    # Randomly select 100 images
    selected_files = selection_rng.sample(
        image_files,
        IMAGES_PER_CLASS
    )

    # Copy the selected images
    for image_path in selected_files:
        shutil.copy2(
            image_path,
            destination_path / image_path.name
        )

    print(f"{class_name}: {len(selected_files)} images selected")

In [ ]:
# Author: Fatema
# Cell 4.0 - Create Dataset Table

data_records = []

for class_name in EXPECTED_CLASSES:
    class_path = SELECTED_PATH / class_name

    for image_path in sorted(class_path.iterdir()):
        if image_path.suffix.lower() in IMAGE_EXTENSIONS:
            data_records.append({
                "image_path": str(image_path),
                "label": class_name
            })

data_df = pd.DataFrame(data_records)

# Numeric label for every image: 0 = Healthy, 1 = Early Blight, 2 = Late Blight
LABEL_MAP = {folder: i for i, folder in enumerate(EXPECTED_CLASSES)}
data_df["label_id"] = data_df["label"].map(LABEL_MAP)

print("Total images:", len(data_df))
print("\nClass counts:")
print(data_df["label"].value_counts())

In [ ]:
# Author: Fatema
# Cell 4.0 - Image Quality and Size Check

image_sizes = set()
image_modes = set()
corrupted_images = []

for image_path in data_df["image_path"]:
    try:
        with Image.open(image_path) as img:
            image_sizes.add(img.size)
            image_modes.add(img.mode)
            img.verify()
    except Exception:
        corrupted_images.append(image_path)

print("Different image sizes found:", image_sizes)
print("Colour modes found:", image_modes)
print("Number of corrupted images:", len(corrupted_images))
print("Missing labels:", data_df["label"].isna().sum())

In [ ]:
# Author: Fatema
# Cell 4.0 - Stratified Split Function and Saved Split

def make_split(df, ratio, seed=SEED):
    # Stratified two-step split. First the training share is taken off, then the
    # rest is divided into validation and test. Every part keeps 1/3 per class.
    train_share, val_share, test_share = ratio
    train_part, rest = train_test_split(
        df, train_size=train_share, stratify=df["label"], random_state=seed)
    val_part, test_part = train_test_split(
        rest, test_size=test_share / (val_share + test_share),
        stratify=rest["label"], random_state=seed)
    return (train_part.reset_index(drop=True),
            val_part.reset_index(drop=True),
            test_part.reset_index(drop=True))


# If the final split was already saved (in 5.4), load it, so that every member and
# every model uses exactly the same images. Otherwise it is created in 5.4.
USE_SAVED_SPLIT = True

manifest_files = {
    "train": SPLIT_PATH / "train_split.csv",
    "validation": SPLIT_PATH / "validation_split.csv",
    "test": SPLIT_PATH / "test_split.csv",
}


def load_manifest(csv_path):
    # Rebuild the full image path from the class label and file name,
    # so the manifest works on any computer
    frame = pd.read_csv(csv_path)
    if "filename" not in frame.columns:
        frame["filename"] = frame["image_path"].apply(lambda p: Path(p).name)
    frame["image_path"] = [
        str(SELECTED_PATH / label / name)
        for label, name in zip(frame["label"], frame["filename"])
    ]
    frame["label_id"] = frame["label"].map(LABEL_MAP)
    return frame[["image_path", "label", "label_id"]].reset_index(drop=True)


def saved_split_matches():
    # A saved split is reused only if it belongs to this dataset's class folders
    try:
        return set(pd.read_csv(manifest_files["train"])["label"]) == set(EXPECTED_CLASSES)
    except Exception:
        return False


SPLIT_LOADED = (USE_SAVED_SPLIT and not CLASSES_TO_BUILD
                and all(path.exists() for path in manifest_files.values())
                and saved_split_matches())

if SPLIT_LOADED:
    train_df = load_manifest(manifest_files["train"])
    val_df = load_manifest(manifest_files["validation"])
    test_df = load_manifest(manifest_files["test"])
    summary_file = SPLIT_PATH / "data_summary.json"
    SAVED_RATIO_NAME = (json.loads(summary_file.read_text()).get("split_ratio", "70/15/15")
                        if summary_file.exists() else "70/15/15")
    print(f"Saved split loaded (ratio {SAVED_RATIO_NAME}): "
          f"{len(train_df)} training / {len(val_df)} validation / {len(test_df)} test images")
else:
    print("No saved split yet: the split ratio is chosen in 5.3 and the final split is made in 5.4.")
    print("WARNING: the official split (80/10/10) was NOT loaded, so a NEW split would be made.\n"
          "Run Cell 1.0 again: it downloads the official split from GitHub automatically.")

### 4.0 Preprocessing pipeline

Every image goes through the same steps:
1. **Decode** as a 3-channel RGB image.
2. **Resize** from 256 × 256 to 224 × 224 (the standard input size of ImageNet
   pre-trained networks, so the baseline and improved model can share this pipeline).
3. **Normalise** pixel values from 0–255 to 0–1.
4. **Cache** the processed images in memory (the dataset is small), then
   **shuffle** (training only), **batch** and **prefetch**.

The function takes the batch size as an argument so the batch-size experiment
uses exactly the same pipeline.

In [ ]:
# Author: Fatema
# Cell 4.0 - Image Preprocessing and Dataset Creation

# Image and training settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

AUTOTUNE = tf.data.AUTOTUNE


# Load, resize and normalize one image
def load_and_preprocess(image_path, label):

    # Read the image file
    image = tf.io.read_file(image_path)

    # Decode it as an RGB image
    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    # Resize to 224 x 224
    image = tf.image.resize(image, IMG_SIZE)

    # Convert pixels from 0-255 to 0-1
    image = tf.cast(image, tf.float32) / 255.0

    return image, label


# Create a TensorFlow dataset
def make_dataset(df, training=False, batch_size=BATCH_SIZE):

    image_paths = df["image_path"].values
    labels = df["label_id"].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    dataset = dataset.map(
        load_and_preprocess,
        num_parallel_calls=AUTOTUNE
    )

    # Keep the decoded images in memory after the first epoch
    dataset = dataset.cache()

    # Shuffle training data only
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


# Check one batch (the training, validation and test datasets are made with this
# same function in 5.3 - 5.8, once the split is fixed)
images, labels = next(iter(make_dataset(data_df, training=True)))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Pixel range:",
      float(tf.reduce_min(images)),
      "to",
      float(tf.reduce_max(images)))

print("Label mapping:", LABEL_MAP)

### 4.0 Data augmentation

Augmentation is implemented as Keras preprocessing layers placed at the start
of the model. Keras only applies these random layers during training
(`training=True`); during validation, testing and prediction they pass images
through unchanged. Validation and test images are therefore never augmented.

A **function** creates the augmentation block so that every model gets a fresh
copy whose randomness is controlled by the seed.

In [ ]:
# Author: Fatema
# Cell 4.0 - Data Augmentation

def make_augmentation():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),   # up to +/-10% of a full turn (about +/-36 degrees)
        tf.keras.layers.RandomZoom(0.1)        # zoom in or out by up to 10%
    ], name="data_augmentation")


# One copy used only for the preview below
data_augmentation = make_augmentation()

In [ ]:
# Author: Fatema
# Cell 4.0 - Augmentation Preview

sample_images, sample_labels = next(iter(make_dataset(data_df, training=True)))

sample_image = sample_images[0]

plt.figure(figsize=(10, 10))

for i in range(9):
    augmented_image = data_augmentation(
        tf.expand_dims(sample_image, 0),
        training=True
    )

    plt.subplot(3, 3, i + 1)
    plt.imshow(augmented_image[0])
    plt.axis("off")

plt.suptitle("Examples of Training Data Augmentation")
plt.tight_layout()
plt.savefig(FIGURES_PATH / "fig3_5_augmentation.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.0 The complete data pipeline

The diagram below shows every step from the Kaggle download to the three
datasets used for training, validation and testing, and which cell does it.

In [ ]:
# Author: Fatema
# Cell 4.0 - Data Pipeline Diagram (Figure 3.4)

from matplotlib.patches import FancyBboxPatch

steps = [
    ("Kaggle download (PlantVillage tomato)", "Healthy, Early Blight, Late Blight folders found (2.0)"),
    ("EDA + class-choice evidence", "imbalance, samples, lesion colour (3.0)"),
    ("Balanced subset", f"{IMAGES_PER_CLASS} per class, seed {SEED} = {len(data_df)} images (4.0)"),
    ("Quality check", f"{len(corrupted_images)} corrupted, sizes {sorted(image_sizes)} (4.0)"),
    ("Split-ratio experiment", "70/15/15 vs 80/10/10 vs 60/20/20; re-checked without test images (5.3)"),
    ("Final stratified split", "chosen ratio, seed 42, saved as manifests (5.4)"),
    ("Leakage checks", "exact file overlap + near-duplicate hash (5.4)"),
    ("Preprocessing (tf.data)", "decode RGB -> resize 224x224 -> /255 -> cache -> batch"),
    ("Augmentation - training only", "flip, rotation 0.1, zoom 0.1 (inside the model)"),
    ("Models", "baseline CNN (5.0); improved model (6.0)"),
]

fig, ax = plt.subplots(figsize=(9, 11))
ax.set_xlim(0, 10)
ax.set_ylim(0, len(steps) * 1.1)
ax.axis("off")
for k, (title, detail) in enumerate(steps):
    y = (len(steps) - 1 - k) * 1.1 + 0.15
    colour = "#dbeafe" if k < 4 else "#dcfce7" if k < 7 else "#fef3c7"
    ax.add_patch(FancyBboxPatch((1, y), 8, 0.8, boxstyle="round,pad=0.02", fc=colour, ec="#334155"))
    ax.text(5, y + 0.52, title, ha="center", va="center", fontsize=11, weight="bold")
    ax.text(5, y + 0.22, detail, ha="center", va="center", fontsize=9)
    if k < len(steps) - 1:
        ax.annotate("", xy=(5, y - 0.28), xytext=(5, y - 0.01),
                    arrowprops=dict(arrowstyle="->", color="#334155"))
ax.set_title("Data Pipeline: from Kaggle Download to Model Input", fontsize=13)
plt.savefig(FIGURES_PATH / "fig3_4_pipeline.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.0 Baseline CNN Model

A custom Convolutional Neural Network (CNN) trained from scratch is used as the
baseline model. It learns visual features from tomato leaf images using
convolution and pooling layers before classifying each image as Healthy,
Early Blight, or Late Blight.

### 5.1 Architecture

- **Three convolution blocks** (32 → 64 → 128 filters, 3 × 3 kernels, ReLU).
  Early blocks detect simple patterns such as edges and colour changes; later
  blocks combine them into lesion and texture patterns.
- **Max-pooling** after each block halves the height and width, which reduces
  computation and makes features less sensitive to small shifts.
- **Global average pooling** turns the final 26 × 26 × 128 feature maps into a
  128-value vector. A `Flatten` layer would produce 86,528 values and about
  5.5 million weights in the next dense layer, which is far too many for
  about 200 training images.
- **Dense (64, ReLU)** combines the features, and **Dense (3, softmax)** outputs
  one probability per class.

The architecture is written as a function with the kernel size as an argument,
so the hyperparameter experiments in 5.5 build exactly the same network.
Layers are named (`conv1` … `output_softmax`) so they can be referred to
directly, e.g. `conv3` is the last convolution layer used for Grad-CAM.

In [ ]:
# Author: Fatema
# Cell 5.1 - Baseline CNN Architecture

def build_baseline_cnn(kernel_size=(3, 3)):

    model = tf.keras.Sequential([

        # Input images
        tf.keras.layers.Input(shape=(224, 224, 3)),

        # Apply augmentation only during training
        make_augmentation(),

        # First convolution block
        tf.keras.layers.Conv2D(
            32,
            kernel_size=kernel_size,
            activation="relu",
            name="conv1"
        ),
        tf.keras.layers.MaxPooling2D(name="pool1"),

        # Second convolution block
        tf.keras.layers.Conv2D(
            64,
            kernel_size=kernel_size,
            activation="relu",
            name="conv2"
        ),
        tf.keras.layers.MaxPooling2D(name="pool2"),

        # Third convolution block (last conv layer - used for Grad-CAM)
        tf.keras.layers.Conv2D(
            128,
            kernel_size=kernel_size,
            activation="relu",
            name="conv3"
        ),
        tf.keras.layers.MaxPooling2D(name="pool3"),

        # Convert feature maps into a smaller feature vector
        tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool"),

        # Fully connected layer
        tf.keras.layers.Dense(
            64,
            activation="relu",
            name="dense_64"
        ),

        # Final prediction layer
        tf.keras.layers.Dense(
            3,
            activation="softmax",
            name="output_softmax"
        )
    ], name="baseline_cnn")

    return model


build_baseline_cnn().summary()

### 5.2 Training setup and overfitting control

| Setting | Value |
|---|---|
| Loss | Sparse categorical cross-entropy (integer labels, 3 classes) |
| Optimiser | Adam (learning rate tuned: 0.0001 / **0.001** / 0.003) |
| Batch size | Tuned: 8 / **16** / 32 |
| Kernel size | Tuned: **3 × 3** / 5 × 5 |
| Maximum epochs | 50 |
| Early stopping | `monitor="val_loss"`, `patience=5`, `restore_best_weights=True` |
| Seed | 42, reset before every model is built |

**Why early stopping?** In a first round of experiments with a *fixed* number of
epochs (10 / 15 / 20), the best result came from the largest value tried
(20 epochs) while the model was still improving, so the best training length had
not been found. A fixed epoch budget is also unfair to settings that learn more
slowly: a smaller learning rate, or a bigger batch (fewer weight updates per epoch),
loses simply because it runs out of time. With early stopping, every run trains
until the validation loss has not improved for 5 epochs (maximum 50), and the
weights from the best epoch are restored. The number of epochs is therefore
chosen fairly for every configuration and reported as **Best epoch**.

**Regularisation:** the baseline's only regulariser is the training-time data
augmentation (flip, rotation, zoom). It has no dropout and no weight decay. Early
stopping is used here to choose the training length fairly. Further overfitting
control (dropout, regularisation, fine-tuning schedule) is part of the improved
model in Section 6.

**Validation set only:** early stopping, model selection and every tuning
decision use the **validation set only**. The test set is used once, in 5.8, after
the final configuration is fixed.

The helper `run_experiment()` below trains one fresh, seeded model with these
settings. It is used for the split-ratio experiment (5.3) and for every
hyperparameter experiment (5.5), so all runs are trained in exactly the same way.

In [ ]:
# Author: Fatema
# Cell 5.2 - Training Helper Functions (early stopping + experiment runner)

import hashlib

MAX_EPOCHS = 50
PATIENCE = 5

# Reuse switch. GPU training is not bit-for-bit repeatable, so "Restart and run all" would
# otherwise change the reported numbers slightly. True: a numbered experiment (5.5) is loaded
# from results/baseline_runs.json when a saved run with exactly the same settings and images
# exists. A changed setting (e.g. a new learning rate in the live demo) is always trained.
# False: every experiment is trained again and the saved file is overwritten.
M1_REUSE_SAVED = True
BASELINE_RUNS_FILE = RESULTS_PATH / "baseline_runs.json"
HARDWARE = "T4 GPU" if tf.config.list_physical_devices("GPU") else "CPU"


class SavedHistory:
    # Stands in for a Keras History object when a run is loaded from file
    def __init__(self, history):
        self.history = history


def images_id(frame):
    # Short fingerprint of which images are in a split (class folder + file name)
    names = sorted(f"{label}/{Path(p).name}" for p, label in zip(frame["image_path"], frame["label"]))
    return hashlib.md5("\n".join(names).encode()).hexdigest()


def load_saved_runs():
    return json.loads(BASELINE_RUNS_FILE.read_text()) if BASELINE_RUNS_FILE.exists() else {}


class KeepBestWeights(tf.keras.callbacks.Callback):
    # Remembers the weights of the epoch with the lowest validation loss and puts
    # them back at the end of training - also if all 50 epochs run without stopping
    def on_train_begin(self, logs=None):
        self.best_val_loss = np.inf
        self.best_weights = None
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        if logs["val_loss"] < self.best_val_loss:
            self.best_val_loss = logs["val_loss"]
            self.best_weights = self.model.get_weights()
            self.best_epoch = epoch + 1

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


def run_experiment(experiment, parameter, value,
                   batch_size=16, learning_rate=0.001, kernel_size=(3, 3),
                   train_data=None, val_data=None, seed=SEED, verbose=2):

    # By default the final split (train_df / val_df from 5.4) is used;
    # the split-ratio experiment (5.3) passes its own candidate split instead
    train_data = train_df if train_data is None else train_data
    val_data = val_df if val_data is None else val_data

    # Settings that define this run; a saved run is reused only if all of them match
    config = {"batch_size": batch_size, "learning_rate": learning_rate, "kernel_size": list(kernel_size),
              "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "seed": seed,
              "train_images": images_id(train_data), "validation_images": images_id(val_data)}
    numbered = isinstance(experiment, int)      # the split-ratio runs ("S") are not saved here
    saved = load_saved_runs().get(str(experiment)) if numbered else None
    if M1_REUSE_SAVED and saved is not None and saved["config"] == config:
        print(f"Experiment {experiment}: loaded the saved run ({saved['hardware']}) - not retrained. "
              "Set M1_REUSE_SAVED = False to train it again.")
        print(saved["summary"])
        return saved["summary"], SavedHistory(saved["history"]), None

    # Same seed, same data pipeline and a fresh model for every experiment
    tf.keras.utils.set_random_seed(seed)
    exp_train_ds = make_dataset(train_data, training=True, batch_size=batch_size)
    exp_val_ds = make_dataset(val_data, batch_size=batch_size)

    model = build_baseline_cnn(kernel_size=kernel_size)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    # Stop when validation loss has not improved for 5 epochs; keep the best weights
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    keep_best = KeepBestWeights()

    start = time.time()
    run_history = model.fit(exp_train_ds, validation_data=exp_val_ds,
                            epochs=MAX_EPOCHS, callbacks=[early_stopping, keep_best],
                            verbose=verbose)
    train_time = time.time() - start

    h = run_history.history
    b = keep_best.best_epoch - 1          # index of the best epoch
    summary = {
        "Exp.": experiment,
        "Parameter varied": parameter,
        "Value": value,
        "Epochs run": len(h["loss"]),
        "Best epoch": keep_best.best_epoch,
        "Train acc.": round(h["accuracy"][b], 4),
        "Val. acc.": round(h["val_accuracy"][b], 4),
        "Train loss": round(h["loss"][b], 4),
        "Val. loss": round(h["val_loss"][b], 4),
        "Params": model.count_params(),
        "Train time (min)": round(train_time / 60, 1),
    }
    print(summary)
    if numbered:
        runs = load_saved_runs()
        runs[str(experiment)] = {"config": config, "summary": summary,
                                 "history": {k: [float(v) for v in vals] for k, vals in h.items()},
                                 "hardware": HARDWARE}
        BASELINE_RUNS_FILE.write_text(json.dumps(runs, indent=1))
    return summary, run_history, model


# Containers for all experiments
experiment_results = []
experiment_histories = {}
experiment_models = {}
experiment_names = {}

### 5.3 Split-ratio experiment: which train / validation / test ratio?

The usual 70 / 15 / 15 ratio is **not assumed to be best**. With only 300
images, the ratio is a real trade-off:

- a **larger training share** (80 / 10 / 10) gives the model more examples to
  learn from, but leaves only 30 validation and 30 test images, so every score
  is noisier (one test image = 3.3 percentage points);
- a **larger evaluation share** (60 / 20 / 20) makes validation and test scores
  more reliable (one test image = 1.7 points), but leaves fewer training images.

**Method (validation set only):**
1. For each ratio, three different random stratified splits are made
   (split seeds 42, 7 and 2024), so the result does not depend on one lucky split.
2. On each split the **reference baseline** (batch 16, learning rate 0.001,
   3 × 3 kernels, early stopping) is trained on the training part and scored on
   the validation part. The test part of every candidate split is **never used**.
   The candidate splits are drawn from all 300 images, independently of the final
   split made in 5.4, so some of their validation images end up in the final test
   set. The check after this experiment counts them and repeats the comparison
   without them.
3. The ratios are compared on the **mean validation loss** over the three splits
   (lower is better); the standard deviation shows how much a ratio depends on
   the split.

**Selection rule (fixed before running):** the ratio with the lowest mean
validation loss wins. If another ratio is within **one standard error** of the
winner (standard error = SD / √3, the "one-standard-error rule" of Hastie et al.,
2009), the difference is treated as noise, and the ratio with the **larger
validation and test sets** is chosen among those, because it gives more reliable
evaluation. The chosen ratio is used for the final split (5.4) and for all
following experiments.

*Note:* this experiment trains 9 models (3 ratios × 3 splits). None of them is
kept; they are only used to choose the ratio.

In [ ]:
# Author: Fatema
# Cell 5.3 - Split-Ratio Experiment (validation set only)

SPLIT_CANDIDATES = {
    "70/15/15": (0.70, 0.15, 0.15),
    "80/10/10": (0.80, 0.10, 0.10),
    "60/20/20": (0.60, 0.20, 0.20),
}
SPLIT_SEEDS = [42, 7, 2024]    # three different random splits per ratio
split_comparison_csv = RESULTS_PATH / "split_ratio_comparison.csv"

if SPLIT_LOADED and split_comparison_csv.exists():
    # The final split already exists (loaded in 4.0): show the saved comparison only
    print("Saved split loaded - showing the saved split-ratio comparison (not re-run).")
    split_runs_df = pd.read_csv(split_comparison_csv)
elif SPLIT_LOADED:
    print("Saved split loaded, but no saved comparison found - the experiment is skipped.")
    split_runs_df = None
else:
    split_runs = []
    for ratio_name, ratio in SPLIT_CANDIDATES.items():
        for split_seed in SPLIT_SEEDS:
            cand_train, cand_val, _ = make_split(data_df, ratio, seed=split_seed)   # test part unused
            print(f"{ratio_name} | split seed {split_seed} | "
                  f"{len(cand_train)} training / {len(cand_val)} validation images")
            summary, _, _ = run_experiment("S", "Split ratio", ratio_name,
                                           train_data=cand_train, val_data=cand_val,
                                           seed=split_seed, verbose=0)
            split_runs.append({
                "Ratio": ratio_name,
                "Split seed": split_seed,
                "Train images": len(cand_train),
                "Validation images": len(cand_val),
                "Best epoch": summary["Best epoch"],
                "Train acc.": summary["Train acc."],
                "Val. acc.": summary["Val. acc."],
                "Val. loss": summary["Val. loss"],
            })
    split_runs_df = pd.DataFrame(split_runs)
    split_runs_df.to_csv(split_comparison_csv, index=False)

if split_runs_df is not None:
    display(split_runs_df)

    # Mean and standard deviation over the three splits of each ratio
    split_summary = split_runs_df.groupby("Ratio", sort=False).agg(
        train_images=("Train images", "first"),
        val_images=("Validation images", "first"),
        mean_val_loss=("Val. loss", "mean"),
        sd_val_loss=("Val. loss", "std"),
        mean_val_acc=("Val. acc.", "mean"),
        sd_val_acc=("Val. acc.", "std"),
        mean_best_epoch=("Best epoch", "mean"),
    ).round(4)
    display(split_summary)

    # Selection rule (validation only): lowest mean validation loss; ratios within
    # one standard error of it count as a tie -> prefer larger validation/test sets
    split_summary["se_val_loss"] = (split_summary["sd_val_loss"] / np.sqrt(len(SPLIT_SEEDS))).round(4)
    best_ratio = split_summary["mean_val_loss"].idxmin()
    tie_limit = split_summary.loc[best_ratio, "mean_val_loss"] + split_summary.loc[best_ratio, "se_val_loss"]
    tied = [r for r in split_summary.index if split_summary.loc[r, "mean_val_loss"] <= tie_limit]
    chosen = max(tied, key=lambda r: split_summary.loc[r, "val_images"])
    if not SPLIT_LOADED:
        CHOSEN_RATIO_NAME = chosen

    print(f"Lowest mean validation loss: {best_ratio} "
          f"({split_summary.loc[best_ratio, 'mean_val_loss']:.4f}, SD {split_summary.loc[best_ratio, 'sd_val_loss']:.4f}, "
          f"SE {split_summary.loc[best_ratio, 'se_val_loss']:.4f})")
    print("Ratios within one standard error of it (treated as a tie):", tied)
    print("Chosen ratio:", chosen)

    # Figure 3.6: mean ± SD of validation loss and accuracy, with the three splits as dots
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    ratios = list(split_summary.index)
    for ax, column, mean_col, sd_col, title in [
            (axes[0], "Val. loss", "mean_val_loss", "sd_val_loss", "Validation Loss (lower is better)"),
            (axes[1], "Val. acc.", "mean_val_acc", "sd_val_acc", "Validation Accuracy")]:
        colours = ["#2563eb" if r == chosen else "#94a3b8" for r in ratios]
        ax.bar(ratios, split_summary[mean_col], yerr=split_summary[sd_col], capsize=6, color=colours)
        for k, r in enumerate(ratios):
            values = split_runs_df.loc[split_runs_df["Ratio"] == r, column]
            ax.scatter([k] * len(values), values, color="black", zorder=3, s=18)
        ax.set_title(title)
        ax.set_ylabel("Mean ± SD (dots = the 3 splits)")
        ax.set_xlabel("Train / validation / test ratio")
        ax.grid(True, axis="y")
    fig.suptitle(f"Split-Ratio Experiment (3 random splits each; chosen: {chosen})")
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / "fig3_6_split_ratio_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    split_summary = None

if SPLIT_LOADED:
    CHOSEN_RATIO_NAME = SAVED_RATIO_NAME
print("Ratio used for the final split:", CHOSEN_RATIO_NAME)

**Interpretation of the split-ratio experiment**

| Ratio | Training / validation images | Mean validation loss ± SD | Mean validation accuracy ± SD |
|---|---|---|---|
| 70/15/15 | 210 / 45 | 0.4581 ± 0.0044 | 79.26% ± 1.28% |
| **80/10/10** | **240 / 30** | **0.4097 ± 0.0353** | **80.00% ± 0.00%** |
| 60/20/20 | 180 / 60 | 0.4588 ± 0.0281 | 81.11% ± 3.47% |

- **80/10/10 had the lowest mean validation loss** (0.4097). Its standard error is
  0.0353 / √3 = 0.020, so the tie limit is 0.430. Both other ratios (0.458 and
  0.459) lie above it, so the difference counts as real and **80/10/10 is chosen**.
- 70/15/15 and 60/20/20 performed almost the same. 80/10/10 gave the model 30 more
  training images than 70/15/15, and with only 300 images this helped.
- Validation accuracy hardly separates the ratios (79–81%): with 30–60 validation
  images it moves in steps of 1.7–3.3 points, which is why the loss was used.
- **Trade-off:** the test set is now 30 images (10 per class), so one test image
  is worth 3.3 percentage points. The bootstrap intervals in 5.8 show this uncertainty.

**Decision based on validation data, with one caveat.** The candidate splits were drawn
independently of the final split, so their validation parts contain up to 15 of the 30
images that later became the final test set (counted in the next cell). No model was trained,
tuned or selected on the final test set itself, so the overlap could only have influenced the
choice of ratio. The next cell checks that the choice does not depend on those images.

### 5.3 Check: does the ratio choice hold without the final test images?

The candidate splits above were drawn from all 300 images, so their validation parts
overlap with the final test set of 5.4. To make sure the ratio was not chosen with the
help of test images, this cell

1. **counts** how many final-test images were in each candidate validation set, and
2. **repeats the comparison on the other 270 images** (the final test set removed
   first): same three ratios, same split seeds, same reference baseline and the same
   selection rule.

If the same ratio wins, the final test set had no influence on the choice. The runs are
saved in `results/split_ratio_check_without_test.csv` and reused when `M1_REUSE_SAVED`
is True.


In [ ]:
# Author: Fatema
# Cell 5.3 - Check: Final-Test Images in the Candidate Splits, and the Ratio Choice Without Them

def image_keys(frame):
    return set(f"{label}/{Path(p).name}" for p, label in zip(frame["image_path"], frame["label"]))

# The final test set: loaded from the manifests, or made now exactly as 5.4 will make it
if SPLIT_LOADED:
    final_test_df = test_df
else:
    _, _, final_test_df = make_split(data_df, SPLIT_CANDIDATES[CHOSEN_RATIO_NAME], seed=SEED)
final_test_keys = image_keys(final_test_df)

# 1. Overlap of every candidate validation set with the final test set
overlap_rows = []
for ratio_name, ratio in SPLIT_CANDIDATES.items():
    for split_seed in SPLIT_SEEDS:
        _, cand_val, _ = make_split(data_df, ratio, seed=split_seed)
        overlap_rows.append({"Ratio": ratio_name, "Split seed": split_seed,
                             "Validation images": len(cand_val),
                             "Of these in the final test set": len(image_keys(cand_val) & final_test_keys)})
overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df)
print("Largest overlap:", overlap_df["Of these in the final test set"].max(), "of", len(final_test_df), "test images")

# 2. Repeat the comparison on the 270 images that are not in the final test set
pool_df = data_df[~data_df.apply(lambda r: f"{r['label']}/{Path(r['image_path']).name}" in final_test_keys,
                                 axis=1)].reset_index(drop=True)
print(f"\nPool without the final test images: {len(pool_df)} images")
split_check_csv = RESULTS_PATH / "split_ratio_check_without_test.csv"

if M1_REUSE_SAVED and split_check_csv.exists():
    print("Saved check loaded (not re-run). Set M1_REUSE_SAVED = False to run it again.")
    check_runs_df = pd.read_csv(split_check_csv)
else:
    check_runs = []
    for ratio_name, ratio in SPLIT_CANDIDATES.items():
        for split_seed in SPLIT_SEEDS:
            cand_train, cand_val, _ = make_split(pool_df, ratio, seed=split_seed)
            assert not (image_keys(cand_train) | image_keys(cand_val)) & final_test_keys
            summary, _, _ = run_experiment("S-check", "Split ratio", ratio_name,
                                           train_data=cand_train, val_data=cand_val,
                                           seed=split_seed, verbose=0)
            check_runs.append({"Ratio": ratio_name, "Split seed": split_seed,
                               "Train images": len(cand_train), "Validation images": len(cand_val),
                               "Best epoch": summary["Best epoch"], "Val. acc.": summary["Val. acc."],
                               "Val. loss": summary["Val. loss"], "Hardware": HARDWARE})
    check_runs_df = pd.DataFrame(check_runs)
    check_runs_df.to_csv(split_check_csv, index=False)

display(check_runs_df)
check_summary = check_runs_df.groupby("Ratio", sort=False).agg(
    train_images=("Train images", "first"), val_images=("Validation images", "first"),
    mean_val_loss=("Val. loss", "mean"), sd_val_loss=("Val. loss", "std"),
    mean_val_acc=("Val. acc.", "mean")).round(4)
check_summary["se_val_loss"] = (check_summary["sd_val_loss"] / np.sqrt(len(SPLIT_SEEDS))).round(4)
display(check_summary)

# Same selection rule as above
check_best = check_summary["mean_val_loss"].idxmin()
check_limit = check_summary.loc[check_best, "mean_val_loss"] + check_summary.loc[check_best, "se_val_loss"]
check_tied = [r for r in check_summary.index if check_summary.loc[r, "mean_val_loss"] <= check_limit]
check_chosen = max(check_tied, key=lambda r: check_summary.loc[r, "val_images"])
SPLIT_CHECK_AGREES = check_chosen == CHOSEN_RATIO_NAME
print("Lowest mean validation loss without test images:", check_best, "| tied:", check_tied)
print("Ratio chosen without test images:", check_chosen,
      "-> same as the chosen ratio" if SPLIT_CHECK_AGREES else f"-> DIFFERENT from {CHOSEN_RATIO_NAME}")


**Result of the check.** The candidate validation sets held 0–15 of the 30 final test images
(largest: 15 of 45 for 70/15/15, seed 42). Without those 30 images, the comparison gives the same
answer:

| Ratio | Training / validation images | Mean validation loss ± SD |
|---|---|---|
| 70/15/15 | 189 / 40 | 0.4402 ± 0.1305 |
| **80/10/10** | **216 / 27** | **0.3591 ± 0.0442** |
| 60/20/20 | 162 / 54 | 0.4312 ± 0.1445 |

80/10/10 again has the lowest mean validation loss, and both alternatives lie above its tie limit
(0.3591 + 0.0442 / √3 = 0.385), so the same rule chooses **80/10/10**. The overlap therefore did not
influence the choice of ratio, and the final test set stays a fair, once-used estimate. (These nine
runs were made on CPU and are saved in `results/split_ratio_check_without_test.csv`; delete that file
to repeat them on the GPU.)


### 5.4 Final split with the chosen ratio, leakage checks and saved manifests

The 300 images are now split **once** with the chosen ratio (stratified, seed 42).
This split is checked for leakage, saved as CSV manifests and used by every
model in the project (baseline and improved), so all models are trained,
tuned and tested on exactly the same images.

In [ ]:
# Author: Fatema
# Cell 5.4 - Final Train, Validation and Test Split (chosen ratio)

if SPLIT_LOADED:
    SPLIT_SOURCE = "loaded from the saved manifests"
else:
    train_df, val_df, test_df = make_split(data_df, SPLIT_CANDIDATES[CHOSEN_RATIO_NAME], seed=SEED)
    SPLIT_SOURCE = f"created with the chosen ratio {CHOSEN_RATIO_NAME} (stratified, seed {SEED})"

print("Split", SPLIT_SOURCE)

# Every manifest image must exist in the selected folder
missing_files = [p for df in (train_df, val_df, test_df) for p in df["image_path"] if not Path(p).exists()]
print("Missing image files:", len(missing_files))

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))

print("\nImages per class in each split:")
display(pd.DataFrame({name: frame["label"].value_counts()
                      for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]}))

In [ ]:
# Author: Fatema
# Cell 5.4 - Data Leakage Check (exact file overlap)

train_paths = set(train_df["image_path"])
val_paths = set(val_df["image_path"])
test_paths = set(test_df["image_path"])

train_val_overlap = train_paths.intersection(val_paths)
train_test_overlap = train_paths.intersection(test_paths)
val_test_overlap = val_paths.intersection(test_paths)

print("Train / Validation overlap:", len(train_val_overlap))
print("Train / Test overlap:", len(train_test_overlap))
print("Validation / Test overlap:", len(val_test_overlap))

The path check above only finds the *same file* in two splits. PlantVillage can
contain several photos of the same leaf saved under different names, which a
path check cannot detect. The next cell compares a small **perceptual hash**
(difference hash) of every image: two images whose hashes differ in only a few
of the 64 bits look almost identical. Any such pair that crosses two splits
would be a leakage risk.

In [ ]:
# Author: Fatema
# Cell 5.4 - Data Leakage Check (near-duplicate images)

def difference_hash(image_path, hash_size=8):
    # Shrink to 9x8 greyscale and compare neighbouring pixels -> 64-bit hash
    with Image.open(image_path) as img:
        small = img.convert("L").resize((hash_size + 1, hash_size))
    pixels = np.asarray(small, dtype=np.int16)
    return (pixels[:, 1:] > pixels[:, :-1]).flatten()

HAMMING_THRESHOLD = 5   # hashes differing in <= 5 of 64 bits = near-duplicate

split_frames = {"train": train_df, "validation": val_df, "test": test_df}
hashes = {
    split: np.array([difference_hash(p) for p in frame["image_path"]])
    for split, frame in split_frames.items()
}

near_duplicate_pairs = []

for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    # Number of differing bits between every image in split a and every image in split b
    distances = (hashes[a][:, None, :] != hashes[b][None, :, :]).sum(axis=2)
    pairs = np.argwhere(distances <= HAMMING_THRESHOLD)
    print(f"{a} / {b} near-duplicate pairs:", len(pairs))

    for i, j in pairs:
        near_duplicate_pairs.append((
            split_frames[a]["image_path"].iloc[i],
            split_frames[b]["image_path"].iloc[j],
            int(distances[i, j])
        ))

# Show any pairs found so they can be inspected
for pair in near_duplicate_pairs[:10]:
    print(pair)

In [ ]:
# Author: Fatema
# Cell 5.4 - Save Reproducible Dataset Splits

def manifest_table(frame, split_name):
    # Portable manifest: file name and class folder instead of a computer-specific path
    return pd.DataFrame({
        "filename": [Path(p).name for p in frame["image_path"]],
        "label": frame["label"].values,
        "label_id": frame["label"].map(LABEL_MAP).values,
        "split": split_name,
        "relative_path": [f"data/selected/{label}/{Path(p).name}"
                          for label, p in zip(frame["label"], frame["image_path"])],
    })


manifest_table(train_df, "train").to_csv(manifest_files["train"], index=False)
manifest_table(val_df, "validation").to_csv(manifest_files["validation"], index=False)
manifest_table(test_df, "test").to_csv(manifest_files["test"], index=False)

print("Split files saved to:", SPLIT_PATH, f"(ratio {CHOSEN_RATIO_NAME})")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

In [ ]:
# Author: Fatema
# Cell 5.4 - Save Data Summary (data_summary.json)

data_summary = {
    "dataset": "plantvillage Tomato leaf dataset (Kaggle, charuchaudhry) - three tomato classes",
    "all_classes_in_download": all_classes,
    "class_folders": EXPECTED_CLASSES,
    "seed": SEED,
    "original_counts": {c: len([f for f in (ORIGINAL_PATH / c).iterdir()
                                if f.suffix.lower() in IMAGE_EXTENSIONS])
                        for c in EXPECTED_CLASSES} if HAS_ORIGINAL else "not available",
    "images_per_class": IMAGES_PER_CLASS,
    "image_sizes": [list(s) for s in image_sizes],
    "colour_modes": sorted(image_modes),
    "corrupted_images": len(corrupted_images),
    "split_ratio": CHOSEN_RATIO_NAME,
    "split_ratio_comparison": (split_summary.reset_index().to_dict(orient="records")
                               if split_summary is not None else "see results/split_ratio_comparison.csv"),
    "split_sizes": {"train": len(train_df), "validation": len(val_df), "test": len(test_df)},
    "split_class_counts": {name: frame["label"].value_counts().to_dict()
                           for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]},
    "exact_overlap": {"train/validation": len(train_val_overlap),
                      "train/test": len(train_test_overlap),
                      "validation/test": len(val_test_overlap)},
    "near_duplicate_pairs": len(near_duplicate_pairs),
    "near_duplicate_threshold_bits": HAMMING_THRESHOLD,
}

with open(SPLIT_PATH / "data_summary.json", "w") as f:
    json.dump(data_summary, f, indent=2)

print(json.dumps(data_summary, indent=2))

### 5.5 Hyperparameter experiments

One hyperparameter is changed at a time; everything else stays as in the
reference. Batch size and learning rate are tested in **both directions**. Kernel
size is tested only upwards: the next smaller option, 1 × 1, cannot see spatial
patterns such as lesions. The number of epochs is no longer fixed, because early
stopping chooses it for every run.

| Exp. | Parameter varied | Value | Everything else |
|---|---|---|---|
| 1 | Reference | – | batch 16, lr 0.001, 3 × 3 kernels |
| 2 | Batch size | 8 | as reference |
| 3 | Batch size | 32 | as reference |
| 4 | Learning rate | 0.0001 | as reference |
| 5 | Learning rate | 0.003 | as reference |
| 6 | Kernel size | 5 × 5 | as reference |

Selection rule: the experiment with the **lowest validation loss** (at its best
epoch) becomes the final baseline.

In [ ]:
# Author: Fatema
# Cell 5.5 - Experiment 1: Reference (batch 16, lr 0.001, 3x3)

result, run_history, run_model = run_experiment(1, "Reference", "bs 16 / lr 0.001 / 3x3")
experiment_results.append(result)
experiment_histories[1] = run_history
experiment_models[1] = run_model
experiment_names[1] = "Exp 1 - reference"

In [ ]:
# Author: Fatema
# Cell 5.5 - Experiment 2: Batch size 8

result, run_history, run_model = run_experiment(2, "Batch size", "8", batch_size=8)
experiment_results.append(result)
experiment_histories[2] = run_history
experiment_models[2] = run_model
experiment_names[2] = "Exp 2 - batch 8"

In [ ]:
# Author: Fatema
# Cell 5.5 - Experiment 3: Batch size 32

result, run_history, run_model = run_experiment(3, "Batch size", "32", batch_size=32)
experiment_results.append(result)
experiment_histories[3] = run_history
experiment_models[3] = run_model
experiment_names[3] = "Exp 3 - batch 32"

In [ ]:
# Author: Fatema
# Cell 5.5 - Experiment 4: Learning rate 0.0001

result, run_history, run_model = run_experiment(4, "Learning rate", "0.0001", learning_rate=0.0001)
experiment_results.append(result)
experiment_histories[4] = run_history
experiment_models[4] = run_model
experiment_names[4] = "Exp 4 - lr 0.0001"

In [ ]:
# Author: Fatema
# Cell 5.5 - Experiment 5: Learning rate 0.003

result, run_history, run_model = run_experiment(5, "Learning rate", "0.003", learning_rate=0.003)
experiment_results.append(result)
experiment_histories[5] = run_history
experiment_models[5] = run_model
experiment_names[5] = "Exp 5 - lr 0.003"

In [ ]:
# Author: Fatema
# Cell 5.5 - Experiment 6: Kernel size 5x5

result, run_history, run_model = run_experiment(6, "Kernel size", "5x5", kernel_size=(5, 5))
experiment_results.append(result)
experiment_histories[6] = run_history
experiment_models[6] = run_model
experiment_names[6] = "Exp 6 - kernel 5x5"

In [ ]:
# Author: Fatema
# Cell 5.6 - Results Table, Validation-Loss Chart and Selection (validation set only)

results_df = pd.DataFrame(experiment_results)
results_df.to_csv(RESULTS_PATH / "baseline_hyperparameter_results.csv", index=False)
display(results_df)

# Validation loss of every experiment; the dot marks the restored best epoch
plt.figure(figsize=(10, 5))
for n, run_history in experiment_histories.items():
    val_losses = run_history.history["val_loss"]
    line, = plt.plot(range(1, len(val_losses) + 1), val_losses, label=experiment_names[n])
    best = int(np.argmin(val_losses))
    plt.scatter(best + 1, val_losses[best], color=line.get_color(), zorder=3)
plt.title("Validation Loss for Each Baseline Experiment (dot = best epoch)")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(FIGURES_PATH / "figD_1_baseline_experiment_val_loss.png", dpi=150, bbox_inches="tight")
plt.show()

# Selection rule: lowest validation loss at the best epoch
best_row = results_df.loc[results_df["Val. loss"].idxmin()]
SELECTED_EXP = int(best_row["Exp."])
print(f"Selected: Experiment {SELECTED_EXP} ({best_row['Parameter varied']} = {best_row['Value']})")
print(f"Best epoch {best_row['Best epoch']}, validation loss {best_row['Val. loss']}, "
      f"validation accuracy {best_row['Val. acc.']:.2%}")

### 5.6 Selected configuration

- **Learning rate mattered most.** 0.0001 learned slowly (best epoch 37, the
  latest of all runs) and stayed behind 0.001 (0.5547 vs 0.4395). 0.003 was
  unstable: training accuracy only 56.25% and the highest validation loss (0.8328).
- **Kernel size:** 5 × 5 was clearly worse (0.6177); 2.6 times as many parameters
  (267,075) are too many to learn from 240 images.
- **Batch size mattered less.** Batch 32 was close to the reference (0.4438 vs
  0.4395). Batch 8 gave the lowest validation loss (**0.3779**, epoch 19) and the
  highest validation accuracy (90.00%), because it makes 30 smaller, noisier updates
  per epoch instead of 15, which helps on a small dataset. With 30 validation
  images, one image = 3.3 points, so the size of this gain is uncertain.
- Every run stopped early, before the 50-epoch limit, so each setting had a fair
  training length.

**Selected: Experiment 2 – batch 8, learning rate 0.001, 3 × 3 kernels.**

**Decision based on the validation set only.** The test set had no influence on
any choice above.

### 5.7 Final baseline

The selected experiment's model, with the weights of its best epoch restored, is
the **final baseline**. It is not retrained, so it is exactly the model shown in
the table. The next cell plots its learning curves, confirms its validation
loss, and saves it.

In [ ]:
# Author: Fatema
# Cell 5.7 - Final Baseline: Learning Curves and Save

final_model_file = RESULTS_PATH / "baseline_cnn_final.keras"
BASELINE_LOADED = experiment_models[SELECTED_EXP] is None     # True when 5.5 loaded the saved runs
if BASELINE_LOADED:
    # Same model as in the report: the one saved by the original training run
    baseline_model = tf.keras.models.load_model(str(final_model_file))
    print("Final baseline loaded from:", final_model_file)
else:
    baseline_model = experiment_models[SELECTED_EXP]
final_history = experiment_histories[SELECTED_EXP].history
final_best_epoch = int(best_row["Best epoch"])
epochs_axis = range(1, len(final_history["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in [(axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")]:
    ax.plot(epochs_axis, final_history[metric], label=f"Training {title}")
    ax.plot(epochs_axis, final_history[f"val_{metric}"], label=f"Validation {title}")
    ax.axvline(final_best_epoch, color="grey", linestyle="--", label=f"Best epoch ({final_best_epoch})")
    ax.set_title(f"Final Baseline CNN - Training and Validation {title}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "fig5_1_final_baseline_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Check: the restored model gives the best-epoch validation loss
val_loss_check, val_acc_check = baseline_model.evaluate(make_dataset(val_df), verbose=0)
print(f"Validation loss {val_loss_check:.4f} | validation accuracy {val_acc_check:.2%}")

if not BASELINE_LOADED:
    baseline_model.save(str(final_model_file))
    print("Final baseline saved to:", final_model_file)

## 5.8 Baseline Evaluation on the Test Set

The final baseline is evaluated **once** on the held-out test set (the test part
of the final split from 5.4, equal numbers per class). Accuracy alone is not enough: per-class precision, recall and
F1 show *which* class fails, the confusion matrix shows *where* the mistakes go,
and ROC-AUC shows how well the predicted probabilities rank the true class.

In [ ]:
# Author: Fatema
# Cell 5.8 - Baseline Test-Set Evaluation

from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score

test_ds = make_dataset(test_df)

test_loss, test_accuracy = baseline_model.evaluate(test_ds, verbose=0)
test_probabilities = baseline_model.predict(test_ds, verbose=0)
y_pred = np.argmax(test_probabilities, axis=1)
y_true = test_df["label_id"].values

macro_f1 = f1_score(y_true, y_pred, average="macro")
macro_auc = roc_auc_score(y_true, test_probabilities, multi_class="ovr")

print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy, 4))
print("Macro F1:", round(macro_f1, 4))
print("Macro ROC-AUC (one-vs-rest):", round(macro_auc, 4))
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap="Blues")
plt.title("Final Baseline CNN - Test Set Confusion Matrix")
plt.savefig(FIGURES_PATH / "fig5_4_baseline_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Author: Fatema
# Cell 5.8 - Error Summary: Where Do the Mistakes Go?

for i, true_name in enumerate(CLASS_NAMES):
    n_class = cm[i].sum()
    print(f"{true_name}: {cm[i, i]}/{n_class} correct (recall {cm[i, i] / n_class:.1%})")
    for j, predicted_name in enumerate(CLASS_NAMES):
        if j != i and cm[i, j] > 0:
            print(f"    {cm[i, j]} predicted as {predicted_name}")

print(f"\nOne test image = {100 / len(y_true):.1f} percentage points of accuracy.")

In [ ]:
# Author: Fatema
# Cell 5.8 - Bootstrap 95% Confidence Intervals (test-set size)

# Resample the test predictions with replacement 1,000 times
rng = np.random.default_rng(SEED)
boot_accuracy, boot_f1 = [], []
for _ in range(1000):
    idx = rng.integers(0, len(y_true), len(y_true))
    boot_accuracy.append(accuracy_score(y_true[idx], y_pred[idx]))
    boot_f1.append(f1_score(y_true[idx], y_pred[idx], average="macro", zero_division=0))

acc_ci = np.percentile(boot_accuracy, [2.5, 97.5])
f1_ci = np.percentile(boot_f1, [2.5, 97.5])
print(f"Test accuracy {test_accuracy:.2%}  (95% CI {acc_ci[0]:.2%} - {acc_ci[1]:.2%})")
print(f"Macro F1      {macro_f1:.2%}  (95% CI {f1_ci[0]:.2%} - {f1_ci[1]:.2%})")

# Save every final number for the team, the README and the report
per_class_recall = recall_score(y_true, y_pred, average=None)
baseline_results = {
    "selected_experiment": SELECTED_EXP,
    "configuration": {k: str(best_row[k]) for k in ["Parameter varied", "Value", "Best epoch", "Epochs run"]},
    "validation_accuracy": float(best_row["Val. acc."]),
    "validation_loss": float(best_row["Val. loss"]),
    "test_loss": round(float(test_loss), 4),
    "test_accuracy": round(float(test_accuracy), 4),
    "test_accuracy_95ci": [round(float(x), 4) for x in acc_ci],
    "macro_f1": round(float(macro_f1), 4),
    "macro_f1_95ci": [round(float(x), 4) for x in f1_ci],
    "macro_roc_auc": round(float(macro_auc), 4),
    "recall_per_class": {name: round(float(r), 4) for name, r in zip(CLASS_NAMES, per_class_recall)},
    "confusion_matrix": cm.tolist(),
    "parameters": int(baseline_model.count_params()),
}
with open(RESULTS_PATH / "baseline_test_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)
print("\nSaved results/baseline_test_results.json")

### 5.8 Interpretation of the test results

- **Overall:** test accuracy **80.00%** (95% CI 66.67–93.33%), macro F1 **79.38%**,
  ROC-AUC **0.8950**. Validation accuracy was 90.00%; the gap is 3 images. The
  validation set was used for early stopping, the split ratio and the choice among
  six settings, so its score is optimistic; the test set (used once) is the honest
  estimate.
- **Per class:** Healthy 10/10, Early Blight 8/10, **Late Blight 6/10** (weakest).
- **Where the errors go:** 3 Late Blight leaves were predicted as Early Blight, the
  confusion expected from the EDA (both have brown lesions, Figure 3.3). 3 diseased
  leaves were predicted as Healthy (2 Early Blight, 1 Late Blight).
- **Costliest error:** a diseased leaf called Healthy, because the farmer would not
  treat the plant. The Late Blight case fits the colour analysis in 3.0: small,
  pale lesions on a mostly green leaf. Grad-CAM (Section 9.0) should check where
  the model looks.
- **Good news:** no healthy leaf was flagged as diseased, and every Late Blight
  prediction was correct (precision 100%).

**Test-set size:** with 30 test images, one image = 3.3 percentage points, so
small differences between models may be noise. The bootstrap 95% confidence
intervals above show how wide this uncertainty is.

### 5.8 Predict a single new image (live demonstration)

Classifies any leaf image (for example one chosen by the tutor) with the final
baseline. The image goes through the **same** `load_and_preprocess` function as
the training data. The example below uses a test image that the model
classifies correctly.

In [ ]:
# Author: Fatema
# Cell 5.8 - Predict a Single Image (live demo)

def predict_leaf(image_path, model=baseline_model):
    # Same preprocessing as training: decode RGB, resize 224x224, scale to 0-1
    image, _ = load_and_preprocess(tf.constant(str(image_path)), 0)
    probabilities = model.predict(tf.expand_dims(image, 0), verbose=0)[0]
    predicted = int(np.argmax(probabilities))

    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(f"Predicted: {CLASS_NAMES[predicted]} ({probabilities[predicted]:.1%})")
    plt.axis("off")
    plt.show()

    for name, p in zip(CLASS_NAMES, probabilities):
        print(f"{name:13s} {p:.2%}")
    return CLASS_NAMES[predicted]


# Example: a correctly classified test image (a Late Blight one if possible)
correct_idx = np.where(y_pred == y_true)[0]
late_blight_correct = [i for i in correct_idx if y_true[i] == 2]
demo_i = (late_blight_correct or list(correct_idx))[0]

print("True class:", CLASS_NAMES[y_true[demo_i]])
predict_leaf(test_df["image_path"].iloc[demo_i])

## 5.9 Member 1 Summary and Handover

**Data:** 300 PlantVillage images (100 Healthy, 100 Early Blight, 100 Late Blight),
stratified **80/10/10** split (240 / 30 / 30) chosen by the split-ratio experiment (5.3), saved in
`data/split_manifests/`, no exact or near-duplicate leakage between splits. The ratio choice
was re-checked without the final test images (5.3).
The class choice is supported by the colour evidence in 3.0 (Figure 3.3).

**Baseline:** 3-block custom CNN (101,699 parameters). Selected configuration:
batch 8, learning rate 0.001, 3 × 3 kernels, early stopping (best epoch 19).
Validation accuracy 90.00%, test accuracy 80.00%, macro F1 79.38%, ROC-AUC 0.8950,
Late Blight recall 60.00%. All values are saved in `results/baseline_test_results.json`.

**Ready for Members 2 and 3** (all defined after running 1.0–5.8):

| Name | What it is |
|---|---|
| `train_df`, `val_df`, `test_df` | The fixed final split (do **not** re-split; ratio in `CHOSEN_RATIO_NAME`) |
| `make_dataset(df, training, batch_size)` | Same preprocessing for every model (output range 0–1) |
| `make_augmentation()` | Same training-only augmentation |
| `CLASS_NAMES`, `LABEL_MAP` | Class order 0 = Healthy, 1 = Early Blight, 2 = Late Blight |
| `baseline_model` | Final baseline (also saved in `results/baseline_cnn_final.keras`) |
| `results_df` | Baseline tuning table |
| `RESULTS_PATH`, `FIGURES_PATH` | Where to save models and figures |

In [ ]:
# Author: Fatema
# Cell 5.9 - Final Numbers and Ready-to-Paste Text (for the report, notebook and README)

from sklearn.metrics import precision_score

pct = lambda x: f"{x * 100:.2f}%"
ref = results_df[results_df["Exp."] == 1].iloc[0]
sel = best_row
sel_time = float(sel["Train time (min)"])
macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
macro_recall = recall_score(y_true, y_pred, average="macro")
per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0)
per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
lines = []
out = lines.append

def by(param, value):
    row = results_df[(results_df["Parameter varied"] == param) & (results_df["Value"] == value)]
    return row.iloc[0] if len(row) else None

# ---------- Q. Class-choice evidence (report 1.1 / 3.2) ----------
out("=== Q. TEXT for report 3.2 (Figure 3.3, class-choice evidence) ===")
lf = {name: lesion_fractions[name].mean() for name in CLASS_NAMES}
out(f"On {N_EVIDENCE} random images per class, the brown/yellow (lesion-coloured) share of the leaf was "
    f"{lf['Healthy']:.1%} for Healthy, {lf['Early Blight']:.1%} for Early Blight and {lf['Late Blight']:.1%} for Late Blight. "
    f"Colour-profile distances: " + "; ".join(f"{k} {v}" for k, v in pair_distances.items()) +
    f" (most similar pair: {closest_pair}). A colour-only classifier reached {colour_accuracy:.1%} "
    f"(correct per class: " + ", ".join(f"{n} {colour_cm[i, i]}/{colour_cm[i].sum()}" for i, n in enumerate(CLASS_NAMES)) +
    f"; Early Blight predicted as Late Blight {colour_cm[1, 2]}x, Late Blight predicted as Healthy {colour_cm[2, 0]}x).")

# ---------- S. Split-ratio experiment (report 3.4) ----------
out("")
out("=== S. TEXT for notebook 5.3 'Interpretation' and report 3.4 (Figure 3.6) ===")
if split_summary is not None:
    parts = [f"{r}: {row['mean_val_loss']:.4f} ± {row['sd_val_loss']:.4f} "
             f"(accuracy {pct(row['mean_val_acc'])} ± {pct(row['sd_val_acc'])}; "
             f"{int(row['train_images'])} training / {int(row['val_images'])} validation images)"
             for r, row in split_summary.iterrows()]
    best_r = split_summary["mean_val_loss"].idxmin()
    reason = ("it had the lowest mean validation loss"
              if CHOSEN_RATIO_NAME == best_r else
              f"it was within one standard error of the lowest ({best_r}), so by the rule fixed in advance "
              f"the ratio with the larger validation and test sets was preferred")
    out(f"Mean validation loss ± SD over three random splits per ratio - " + "; ".join(parts) +
        f". The chosen ratio is {CHOSEN_RATIO_NAME}, because {reason}. "
        f"The final split has {len(train_df)} / {len(val_df)} / {len(test_df)} images. "
        f"The decision used validation data only. The candidate splits were drawn independently of the "
        f"final split, so their validation parts contained up to {overlap_df['Of these in the final test set'].max()} "
        f"of the {len(test_df)} final test images; no model was trained, tuned or selected on the final test set.")
    check_parts = [f"{r}: {row['mean_val_loss']:.4f} ± {row['sd_val_loss']:.4f}" for r, row in check_summary.iterrows()]
    out(f"Check without the final test images ({len(pool_df)} images) - mean validation loss ± SD: "
        + "; ".join(check_parts) + f". Ratio chosen by the same rule: {check_chosen} "
        + ("(the same ratio, so the overlap did not influence the choice)." if SPLIT_CHECK_AGREES
           else f"(different from {CHOSEN_RATIO_NAME}; report this)."))
else:
    out("Split-ratio comparison not available in this session (saved split loaded).")
out("")

# ---------- A. Report Table 4.2 ----------
cfg = {"batch": "16", "lr": "0.001", "kernel": "3x3"}
if sel["Parameter varied"] == "Batch size": cfg["batch"] = sel["Value"]
if sel["Parameter varied"] == "Learning rate": cfg["lr"] = sel["Value"]
if sel["Parameter varied"] == "Kernel size": cfg["kernel"] = sel["Value"]
out("=== A. REPORT TABLE 4.2 (training configuration) ===")
out(f"Learning rate selected: {cfg['lr']} | Batch size selected: {cfg['batch']} | Kernel: {cfg['kernel']}")
out(f"Best epoch: {sel['Best epoch']} (training stopped at epoch {sel['Epochs run']})")
saved_runs = load_saved_runs()
run_hardware = saved_runs.get(str(SELECTED_EXP), {}).get("hardware", HARDWARE)
out(f"Hardware: {run_hardware} | Final training time: {sel_time} min")

# ---------- B. Report Table 5.1 ----------
out("")
out("=== B. REPORT TABLE 5.1 (Exp | Best epoch (of run) | Train acc | Val acc | Val loss | Observation) ===")
for _, r in results_df.iterrows():
    if int(r["Exp."]) == SELECTED_EXP:
        note = "Lowest validation loss - selected"
    elif int(r["Exp."]) == 1:
        note = "Reference"
    else:
        direction = "Lower" if r["Val. loss"] < ref["Val. loss"] else "Higher"
        note = f"{direction} validation loss than the reference ({r['Val. loss']} vs {ref['Val. loss']})"
    if int(r["Epochs run"]) == MAX_EPOCHS:
        note += "; reached the 50-epoch limit"
    out(f"{int(r['Exp.'])} | {r['Best epoch']} ({r['Epochs run']}) | {pct(r['Train acc.'])} | "
        f"{pct(r['Val. acc.'])} | {r['Val. loss']} | {note}")

# ---------- C. Report Table 5.2 ----------
out("")
out("=== C. REPORT TABLE 5.4 (baseline row; Accuracy | Precision | Recall | F1 | AUC | Params | Train time) ===")
out(f"{pct(test_accuracy)} ({pct(acc_ci[0])}-{pct(acc_ci[1])}) | {pct(macro_precision)} | {pct(macro_recall)} | "
    f"{pct(macro_f1)} ({pct(f1_ci[0])}-{pct(f1_ci[1])}) | {macro_auc:.4f} | "
    f"{baseline_model.count_params():,} | {sel_time} min")

# ---------- D. Report Table 5.3 ----------
out("")
out("=== D. REPORT TABLE 5.5 (Class | Precision | Recall | F1 | Correct / test images of the class) ===")
for i, name in enumerate(CLASS_NAMES):
    out(f"{name} | {pct(per_class_precision[i])} | {pct(per_class_recall[i])} | {pct(per_class_f1[i])} | {cm[i, i]}/{cm[i].sum()}")

# ---------- E. Text for notebook 5.6 and report 5.2 ----------
def describe(param, values):
    parts = []
    for v in values:
        row = ref if (param, v) in [("Batch size", "16"), ("Learning rate", "0.001")] else by(param, v)
        if row is not None:
            parts.append(f"{v}: {row['Val. loss']} (epoch {row['Best epoch']})")
    return "; ".join(parts)

k5 = by("Kernel size", "5x5")
out("")
out("=== E. TEXT for notebook 5.6 'Selected configuration' and report 5.2 ===")
out(f"Validation loss at the best epoch - batch size {describe('Batch size', ['8', '16', '32'])}. "
    f"Learning rate {describe('Learning rate', ['0.0001', '0.001', '0.003'])}. "
    f"Kernel 3x3: {ref['Val. loss']} vs 5x5: {k5['Val. loss']} "
    f"({int(k5['Params']):,} instead of {int(ref['Params']):,} parameters). "
    f"The selected configuration is Experiment {SELECTED_EXP} ({sel['Parameter varied']} = {sel['Value']}), "
    f"with the lowest validation loss ({sel['Val. loss']}) at epoch {sel['Best epoch']} and validation accuracy "
    f"{pct(sel['Val. acc.'])}. Early stopping gave every setting a fair training length. "
    f"The decision used the validation set only; the test set had no influence.")

# ---------- F. Text for report 5.1 (learning curves) ----------
h = final_history
b = int(sel["Best epoch"]) - 1
gap = h["accuracy"][b] - h["val_accuracy"][b]
fit = ("no clear overfitting" if abs(gap) < 0.08 else
       "some overfitting (training accuracy clearly higher)" if gap > 0 else
       "validation accuracy above training accuracy, as expected with training-only augmentation")
out("")
out("=== F. TEXT for report 5.1 (Figure 5.1) ===")
start_note = (" - close to ln 3 = 1.0986, the loss of random guessing between three classes"
              if abs(h["val_loss"][0] - 1.0986) < 0.05 else "")
out(f"Training started with validation loss {h['val_loss'][0]:.4f}{start_note}. The best validation loss ({h['val_loss'][b]:.4f}) was reached at epoch {b + 1}; training stopped "
    f"at epoch {len(h['loss'])} after {PATIENCE} epochs without improvement, and the best weights were restored. At the "
    f"best epoch, training accuracy was {pct(h['accuracy'][b])} and validation accuracy {pct(h['val_accuracy'][b])} "
    f"(training loss {h['loss'][b]:.4f}, validation loss {h['val_loss'][b]:.4f}), so there is {fit}.")

# ---------- G. Text for notebook 5.8 interpretation and report 5.3 ----------
recalls = per_class_recall
worst = int(np.argmin(recalls))
wrong = [(cm[worst, j], CLASS_NAMES[j]) for j in range(3) if j != worst and cm[worst, j] > 0]
wrong_text = " and ".join(f"{n} as {name}" for n, name in sorted(wrong, reverse=True)) or "none"
good = [f"{CLASS_NAMES[i]} ({cm[i, i]}/{cm[i].sum()})" for i in range(3) if i != worst]
out("")
out("=== G. TEXT for notebook 5.8 'Interpretation' and report 5.3 ===")
text = (f"The final baseline reached a test accuracy of {pct(test_accuracy)} (95% CI {pct(acc_ci[0])}-{pct(acc_ci[1])}), "
        f"compared with {pct(sel['Val. acc.'])} on validation; macro F1 was {pct(macro_f1)} and macro ROC-AUC {macro_auc:.4f}. "
        f"{' and '.join(good)} are recognised best. The weakest class is {CLASS_NAMES[worst]} "
        f"(recall {pct(recalls[worst])}, {cm[worst, worst]}/{cm[worst].sum()}): {wrong_text}.")
late_to_healthy = cm[2, 0]
if late_to_healthy > 0:
    leaves = "leaf" if late_to_healthy == 1 else "leaves"
    text += (f" Predicting Healthy for a Late Blight leaf ({late_to_healthy} {leaves}) is the costliest error for a "
             f"screening tool, because the farmer would not treat the plant.")
if worst == 2:
    text += (" A likely cause is that both blights produce similar brown, necrotic lesions, and that Late Blight "
             "often shows small, pale lesions near the leaf edge that are easy to miss on a mostly green leaf.")
elif worst == 1:
    text += (" A likely cause is that both blights produce similar brown, necrotic lesions.")
text += (f" With {len(y_true)} test images, one image = {100 / len(y_true):.1f} percentage points, "
         f"so small differences between models may be noise.")
out(text)

# ---------- H. Notebook 5.9 summary ----------
out("")
out("=== H. TEXT for notebook 5.9 summary ===")
out(f"Selected configuration: batch {cfg['batch']}, learning rate {cfg['lr']}, {cfg['kernel']} kernels, early stopping "
    f"(best epoch {sel['Best epoch']}). Validation accuracy {pct(sel['Val. acc.'])}, test accuracy {pct(test_accuracy)}, "
    f"macro F1 {pct(macro_f1)}, ROC-AUC {macro_auc:.4f}, Late Blight recall {pct(recalls[2])}.")

report_text = "\n".join(lines)
print(report_text)
(RESULTS_PATH / "NUMBERS_FOR_REPORT.txt").write_text(report_text)

# Update the README results table automatically
readme = PROJECT_DIR / "README.md"
if readme.exists():
    text_md = readme.read_text()
    new_row = (f"| Baseline CNN (final) | {pct(test_accuracy)} | {pct(macro_f1)} | {macro_auc:.4f} | "
               f"{baseline_model.count_params():,} |")
    import re
    text_md = re.sub(r"\| Baseline CNN \(final\) \|.*\|", new_row, text_md)
    readme.write_text(text_md)
    print("\nREADME.md results table updated.")
print("Saved results/NUMBERS_FOR_REPORT.txt")

In [ ]:
# Author: Fatema
# Cell 5.9 - Download the Updated Project Files (optional)

# The project's results are kept in the GitHub repository, so "Run all" does not download
# anything. Set this to True to get a ZIP of the project (without the original images),
# for example after re-training the baseline in Colab, where storage is temporary.
DOWNLOAD_MEMBER1_RESULTS = False

if DOWNLOAD_MEMBER1_RESULTS:
    EXPORT_ROOT = UPLOAD_DIR / "export"
    if EXPORT_ROOT.exists():
        shutil.rmtree(EXPORT_ROOT)

    # Copy the project without the large original images
    def skip_large(folder, names):
        skip = [n for n in names if n.endswith(".zip")]
        if Path(folder) == PROJECT_DIR / "data":
            skip.append("original")
        return skip

    shutil.copytree(PROJECT_DIR, EXPORT_ROOT / "IT8414_Tomato_Project", ignore=skip_large)
    archive = shutil.make_archive(str(UPLOAD_DIR / "IT8414_Member1_results"), "zip", EXPORT_ROOT)
    shutil.rmtree(EXPORT_ROOT)
    print("ZIP size:", round(Path(archive).stat().st_size / 1e6, 1), "MB")
    if IN_COLAB:
        files.download(archive)
    else:
        print("Offline mode - the ZIP is saved at:", archive)
else:
    print("No download needed: the results are in results/ and in the GitHub repository.")


## 6.0 Improved Model — *Member 2*

*(Member 2: add the transfer-learning model here — 6.1 Architecture, 6.2 Training, 6.3 Curves.)*

Note: `make_dataset` outputs pixels in the range 0–1. MobileNetV2 expects −1 to 1,
so add `tf.keras.layers.Rescaling(2.0, offset=-1.0)` after the augmentation block.
ResNet50 expects 0–255 with its own `preprocess_input`.

## 7.0 Hyperparameter Experiments and Overfitting Control — *Member 2*

*(Member 2: tuning of the improved model — three or more hyperparameters, dropout / early stopping, results table.)*

## 8.0 Test-Set Evaluation — *Member 3*

*(Member 3: final test evaluation of all models — accuracy, precision, recall, F1 per class and macro, confusion matrices, ROC-AUC; one comparison table.
The baseline numbers are already produced in cell 5.8.)*

## 9.0 Interpretability and Error Analysis — *Member 3*

*(Member 3: Grad-CAM on `conv3` of the baseline and the last conv layer of the improved model; misclassified test images and their likely causes.)*

## 10.0 Conclusion — *Member 3*

*(Member 3: summary of findings and deployment recommendation.)*